In [1]:
import os, time, importlib.util, gc
import numpy as np, torch

path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "09_train_student.py" in files:
        path = os.path.join(root, "09_train_student.py"); break
spec = importlib.util.spec_from_file_location("tb", path)
tb = importlib.util.module_from_spec(spec); spec.loader.exec_module(tb)
print("Script:", path, "| GPU:", torch.cuda.get_device_name(0))

def probe(backbone, B, llrd, V=5, D=32, S=224, steps=4):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    tb.CFG.update(llrd=llrd, lr_backbone=1e-4, lr_head=1e-3)
    net = tb.KneeNet(backbone, n_views=V).cuda()
    if "vit" not in backbone:
        net = net.to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(tb.param_groups(net), weight_decay=1e-2)
    scaler = torch.amp.GradScaler("cuda")
    x = torch.rand(B, V, D, S, S, device="cuda")
    vm = torch.ones(B, V, device="cuda")
    y = torch.rand(B, 12, device="cuda"); ym = torch.ones_like(y)
    try:
        ts = []
        for i in range(steps):
            torch.cuda.synchronize(); t0 = time.time()
            with torch.autocast("cuda", dtype=torch.float16):
                loss = tb.masked_bce(net(x, vm).float(), y, ym)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            torch.cuda.synchronize(); ts.append(time.time() - t0)
        res = f"memori puncak {torch.cuda.max_memory_allocated()/1e9:5.1f} GB | {np.mean(ts[1:]):.2f} detik/step"
    except torch.cuda.OutOfMemoryError:
        res = "OOM (kehabisan memori)"
    del net, opt, x
    gc.collect(); torch.cuda.empty_cache()
    return res

for backbone, llrd in [("efficientnet_b0", 1.0), ("vit_small_patch14_dinov2.lvd142m", 0.8)]:
    for B in [1, 2]:
        print(f"{backbone:34s} batch {B}: {probe(backbone, B, llrd)}")

Script: /kaggle/input/notebooks/kellenvalerie/rsna-knee-10-student-v3/09_train_student.py | GPU: Tesla T4


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

efficientnet_b0                    batch 1: memori puncak   7.4 GB | 0.32 detik/step
efficientnet_b0                    batch 2: memori puncak  14.7 GB | 0.66 detik/step


model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

LLRD 0.8: LR backbone 5.50e-06 (bawah) .. 1.00e-04 (atas)
vit_small_patch14_dinov2.lvd142m   batch 1: memori puncak   8.4 GB | 0.59 detik/step
LLRD 0.8: LR backbone 5.50e-06 (bawah) .. 1.00e-04 (atas)
vit_small_patch14_dinov2.lvd142m   batch 2: OOM (kehabisan memori)
